# Benedict - Profitability Classification

This notebook trains a Decision Tree and Random Forest to predict whether a policy is profitable.

Target: `profitable = 1` if `premium > cost_claims_year`, else `0`.

The workflow uses a temporal split: train on 2017-2018 and test on 2019.


In [ ]:
from pathlib import Path
import time

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)


## Load Data

Adjust `DATA_PATH` if your local or Colab folder structure is different.


In [ ]:
DATA_PATH = Path("Project/Dataset of health insurance portfolio/Dataset of health insurance portfolio/Dataset of health insurance portfolio.csv")

if not DATA_PATH.exists():
    # Colab fallback if Drive is mounted and the dataset was copied there.
    DATA_PATH = Path("/content/drive/MyDrive/AML Project/Dataset/Dataset of health insurance portfolio.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}")
print(f"Raw shape: {df.shape}")


## Preprocess

This mirrors the project notebook's S/P product filter, three-year customer filter, train-only imputation, and train-only categorical encoding.


In [ ]:
start = time.time()

df_sp = df[df["type_product"].isin(["S", "P"])]
years_per_id = df_sp.groupby("ID")["period"].nunique()
keep_ids = years_per_id[years_per_id == 3].index
df_project = df_sp[df_sp["ID"].isin(keep_ids)].copy()

keep_cols = [
    "ID", "ID_policy", "ID_insured", "period",
    "date_effect_insured", "date_lapse_insured", "date_effect_policy", "date_lapse_policy",
    "year_effect_insured", "year_lapse_insured", "year_effect_policy", "year_lapse_policy",
    "exposure_time", "lapse", "seniority_insured", "seniority_policy",
    "type_policy", "type_policy_dg", "type_product", "reimbursement",
    "new_business", "distribution_channel", "gender", "age",
    "premium", "cost_claims_year", "n_medical_services",
    "n_insured_pc", "n_insured_mun", "n_insured_prov", "IICIMUN", "IICIPROV",
    "C_H", "C_GI", "C_II", "C_IE_P", "C_IE_S", "C_IE_T",
    "C_GE_P", "C_GE_S", "C_GE_T", "C_C",
]
df_project = df_project[keep_cols].copy()
df_project["unprofitable"] = (df_project["cost_claims_year"] > df_project["premium"]).astype(int)
df_project["profitable"] = (df_project["premium"] > df_project["cost_claims_year"]).astype(int)

train = df_project[df_project["period"].isin([2017, 2018])].copy()
test = df_project[df_project["period"] == 2019].copy()

socio_cols = ["C_GI", "C_II", "C_IE_P", "C_IE_S", "C_IE_T", "C_GE_P", "C_GE_S", "C_GE_T"]
geo_cols = ["C_H", "C_C"]
concentration_cols = ["IICIMUN", "IICIPROV"]

train["geo_missing_flag"] = train[geo_cols].isnull().any(axis=1).astype(int)
test["geo_missing_flag"] = test[geo_cols].isnull().any(axis=1).astype(int)

train_medians = train[socio_cols + concentration_cols].median()
train[socio_cols + concentration_cols] = train[socio_cols + concentration_cols].fillna(train_medians)
test[socio_cols + concentration_cols] = test[socio_cols + concentration_cols].fillna(train_medians)

train_modes = train[geo_cols].mode().iloc[0]
train[geo_cols] = train[geo_cols].fillna(train_modes)
test[geo_cols] = test[geo_cols].fillna(train_modes)

habitat_order = {"H1": 1, "H2": 2, "H3": 3, "H4": 4, "H5": 5, "H6": 6}
train["C_H_encoded"] = train["C_H"].map(habitat_order)
test["C_H_encoded"] = test["C_H"].map(habitat_order)

cat_cols = [
    "type_policy", "type_policy_dg", "type_product", "reimbursement",
    "new_business", "distribution_channel", "gender", "C_C",
]
for col in cat_cols:
    encoder = LabelEncoder()
    encoder.fit(train[col].astype(str))
    train[f"{col}_encoded"] = encoder.transform(train[col].astype(str))
    mapping = {label: code for code, label in enumerate(encoder.classes_)}
    test[f"{col}_encoded"] = test[col].astype(str).map(mapping).fillna(-1).astype(int)


## Leakage Controls

The model excludes direct target columns and likely post-outcome columns. `premium` is also excluded because the target is defined using `premium > cost_claims_year`.


In [ ]:
base_features = [
    "age", "gender_encoded",
    "type_policy_encoded", "type_policy_dg_encoded", "type_product_encoded",
    "reimbursement_encoded", "new_business_encoded", "distribution_channel_encoded",
    "premium", "seniority_insured", "seniority_policy", "exposure_time",
    "n_insured_pc", "n_insured_mun", "n_insured_prov", "IICIMUN", "IICIPROV",
    "C_GI", "C_II", "C_IE_P", "C_IE_S", "C_IE_T", "C_GE_P", "C_GE_S", "C_GE_T",
    "C_H_encoded", "C_C_encoded", "geo_missing_flag",
]

leakage_columns = [
    "cost_claims_year", "unprofitable", "profitable", "n_medical_services",
    "lapse", "date_lapse_insured", "date_lapse_policy",
    "year_lapse_insured", "year_lapse_policy",
]

profit_features = [
    feature for feature in base_features
    if feature != "premium" and feature not in leakage_columns
]

X_train = train[profit_features].copy()
y_train = train["profitable"].copy()
X_test = test[profit_features].copy()
y_test = test["profitable"].copy()

leakage_present = [col for col in leakage_columns if col in X_train.columns]

print(f"Project subset shape: {df_project.shape}")
print(f"Train shape: {X_train.shape}; Test shape: {X_test.shape}")
print(f"Train profitable rate: {y_train.mean() * 100:.2f}%")
print(f"Test profitable rate: {y_test.mean() * 100:.2f}%")
print(f"Leakage columns still present: {leakage_present}")


## Train Decision Tree and Random Forest


In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=100,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=50,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
}

results = []
for name, model in models.items():
    model_start = time.time()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
        "seconds": time.time() - model_start,
    })

    print("\n" + "=" * 80)
    print(name)
    print("Confusion matrix [actual rows: not profitable, profitable]:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["not profitable", "profitable"]))

model_results = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("\nModel comparison:")
print(model_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

rf_model = models["Random Forest"]
feature_importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": rf_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 20 Random Forest feature importances:")
print(feature_importance.head(20).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"\nTotal elapsed seconds: {time.time() - start:.2f}")
